# Tierkreis checkpoints; using pre build graphs

In this example we will run a predefined graph in a customized environment.

By default tierkreis will store the checkpoints of workflow runs in the filesystem at `~/.tierkreis/checkpoints`.
This can be configured in the storage by providing the `tierkreis_directory` argument to the storage layer.


In [1]:
%pip install tierkreis qnexus

/Users/philipp.seitz/Projects/tierkreis/.devenv/state/venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from uuid import UUID
from tierkreis.storage import FileStorage

storage = FileStorage(
    UUID(int=107),
    do_cleanup=True,
    tierkreis_directory=Path.home() / ".tierkreis" / "checkpoints2",
)

This change needs to be forwarded to the executors, which is why they take the logs path as additional argument.


In [3]:
from tierkreis.consts import PACKAGE_PATH
from tierkreis.executor import UvExecutor

executor = UvExecutor(PACKAGE_PATH.parent / "tierkreis_workers", storage.logs_path)

Typically now one would define a graph.
Although, tierkreis ships some preconstructed graphs.
For this example we are going a to run a circuit on nexus and poll for its results.

In [4]:
from tierkreis.graphs.nexus.submit_poll import nexus_submit_and_poll

graph = nexus_submit_and_poll()

Before we run the graph we can visualize it to get familiar with it.
This will start a webserver and listen on [localhost](http://localhost:8000)

In [5]:
from qnexus.client.auth import login
from tierkreis_visualization.visualize_graph import visualize_graph
login()
visualize_graph(graph) # this spawns a server, you need to manually terminate this cell.

🌐 Browser log in initiated.


╭────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                        │
│         Confirm that the browser shows the following code and click 'allow device':    │
│                                                                                        │
│                                      JM9IMT                                            │
│                                                                                        │
╰────────────────────────────────────────────────────────────────────────────────────────╯

Browser didn't open automatically? Use this link: https://nexus.quantinuum.com/auth/device/browser?otp=JM9IMTxchVeK4KI6mKQQfmqpFkQzGh1TZI4BvIBmnd6K6kaA5MIwQvmTNmeUZJEkT0Nd-v6Vd8CYruEh0nE55g


INFO:     Started server process [26940]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ Successfully logged in as philipp.seitz@quantinuum.com using the browser.
INFO:     127.0.0.1:58590 - "GET /api/workflows/ HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [26940]


KeyboardInterrupt: 

and finally we can run it

In [8]:
from qnexus import AerConfig
from pytket.qasm.qasm import circuit_from_qasm

from tierkreis.controller import run_graph
from tierkreis.storage import read_outputs

aer_config = AerConfig()
circuit = circuit_from_qasm(Path().parent / "data" / "ghz_state_n23.qasm")
circuits = [circuit]
inputs = {
    "project_name": "2025-tkr-test",
    "job_name": "job-1",
    "circuits": circuits,
    "n_shots": [30] * len(circuits),
    "backend_config": aer_config,
}
storage.clean_graph_files()
run_graph(
    storage,
    executor,
    graph,
    inputs,
    polling_interval_seconds=0.1,
)
res = read_outputs(graph, storage)
print(res)

KeyboardInterrupt: 